In [8]:
import pandas as pd
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

True

In [9]:
df = pd.read_csv("../data/cleaned_data.csv")

df.head()

,id,title,type,country,release_year,genre,description,text
0,1,The Food Journey,Movie,Sweden,2020,Documentary,A documentary about food culture in Sweden,\n Titel: The Food Journey\n Typ: Movie\...
1,2,Kitchen Secrets,TV Show,USA,2021,Reality,A show about restaurants and cooking,\n Titel: Kitchen Secrets\n Typ: TV Show...
2,3,Nordic Taste,Movie,Denmark,2019,Drama,A story about a chef in Copenhagen,\n Titel: Nordic Taste\n Typ: Movie\n ...
3,4,Hotel Nights,TV Show,Sweden,2022,Drama,A series about life inside a hotel,\n Titel: Hotel Nights\n Typ: TV Show\n ...
4,5,Street Food Stories,Movie,Thailand,2023,Documentary,A documentary about street food and people,\n Titel: Street Food Stories\n Typ: Mov...


In [5]:
documents = []

for index, row in df.iterrows():
    doc = Document(
        page_content=row["text"],
        metadata={
            "row": index,
            "title": row["title"]
        }
    )
    documents.append(doc)

len(documents)

5

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splits = text_splitter.split_documents(documents)

len(splits)

5

In [10]:
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="../vectorstore"
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-test. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [11]:
import unicodedata

def remove_special_chars(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")

for doc in splits:
    doc.page_content = remove_special_chars(doc.page_content)

embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="../vectorstore"
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-test. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [12]:
from langchain_core.embeddings import Embeddings

class SimpleEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [[float(len(text)), float(text.count("food")), float(text.count("hotel"))] for text in texts]

    def embed_query(self, text):
        return [float(len(text)), float(text.count("food")), float(text.count("hotel"))]

embeddings = SimpleEmbeddings()

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="../vectorstore"
)

In [13]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [14]:
question = "Which titles are about food?"

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Dokument {i} ---")
    print(doc.page_content)
    print(doc.metadata)

--- Dokument 1 ---
Titel: Nordic Taste
    Typ: Movie
    Land: Denmark
    Ar: 2019
    Genre: Drama
    Beskrivning: A story about a chef in Copenhagen
{'title': 'Nordic Taste', 'row': 2}
--- Dokument 2 ---
Titel: Hotel Nights
    Typ: TV Show
    Land: Sweden
    Ar: 2022
    Genre: Drama
    Beskrivning: A series about life inside a hotel
{'title': 'Hotel Nights', 'row': 3}
--- Dokument 3 ---
Titel: Kitchen Secrets
    Typ: TV Show
    Land: USA
    Ar: 2021
    Genre: Reality
    Beskrivning: A show about restaurants and cooking
{'title': 'Kitchen Secrets', 'row': 1}


In [15]:
def ask_rag(question):
    retrieved_docs = retriever.invoke(question)

    print("Fråga:")
    print(question)
    print("\nHämtad kontext från vår data:")
    
    for i, doc in enumerate(retrieved_docs, start=1):
        print(f"\n--- Dokument {i} ---")
        print(doc.page_content)

ask_rag("Which titles are about food?")

Fråga:
Which titles are about food?

Hämtad kontext från vår data:

--- Dokument 1 ---
Titel: Nordic Taste
    Typ: Movie
    Land: Denmark
    Ar: 2019
    Genre: Drama
    Beskrivning: A story about a chef in Copenhagen

--- Dokument 2 ---
Titel: Hotel Nights
    Typ: TV Show
    Land: Sweden
    Ar: 2022
    Genre: Drama
    Beskrivning: A series about life inside a hotel

--- Dokument 3 ---
Titel: Kitchen Secrets
    Typ: TV Show
    Land: USA
    Ar: 2021
    Genre: Reality
    Beskrivning: A show about restaurants and cooking
